# 05. Deep Research Agent — Parallel Subagents and a Five-Step Workflow

## Learning Objectives

- Configure three parallel subagents (`researcher-1`, `researcher-2`, `fact-checker`)
- Implement strategic reflection with `think_tool`
- Design a five-step workflow (Plan → Delegate → Synthesize → Verify → Report)
- Apply v1 middleware (`SummarizationMiddleware`, `ModelCallLimitMiddleware`, `ModelFallbackMiddleware`)


## Overview

| Item | Details |
|------|------|
| **Framework** | Deep Agents |
| **Core components** | Three parallel subagents, `think_tool` |
| **Workflow** | 5 steps: Plan → Delegate → Synthesize → Verify → Report |
| **Backend** | `FilesystemBackend(root_dir=".", virtual_mode=True)` |
| **Built-in tools** | `write_todos` (planning), `task` (subagent call) |
| **Skill** | `skills/deep-research/SKILL.md` — research method + citation rules |


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"


In [ ]:
# Optional observability setup
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")


## Step 1: `think_tool` — A Strategic Reflection Tool

`think_tool` lets the agent record “thoughts” before it acts. This pattern can improve the quality of agent decision-making:

- Analyze search results and plan the next action
- Evaluate whether the collected information is sufficient
- Make delegated tasks more specific before sending them to subagents


In [ ]:
from langchain.tools import tool

@tool
def think_tool(thought: str) -> str:
    """Strategic reflection — analyze the current situation and plan the next action."""
    return f"Reflection recorded: {thought}"


## Step 2: A Simplified `web_search` Tool

In a real deep research workflow, you would typically use the Tavily API. Here, for learning purposes, we define a simplified search tool.


In [ ]:
@tool
def web_search(query: str) -> str:
    """Perform a web search (simulated)."""
    results = {
        "AI agent": "AI agents are systems that perform tasks autonomously. Their adoption has accelerated rapidly since 2024.",
        "LangGraph": "LangGraph is a stateful workflow framework. It supports both the Graph API and the Functional API.",
        "Deep Agents": "Deep Agents is an all-in-one agent SDK. It supports subagents, backends, and skills.",
    }
    for key, val in results.items():
        if key.lower() in query.lower():
            return val
    return f"Search result for '{query}': no relevant information found."


## Step 3: Prompt for the Five-Step Research Workflow

The prompt loader pulls the prompt using the following order: LangSmith Hub → Langfuse → local default.

| Step | Name | Description |
|------|------|------|
| 1 | **Plan** | Create a research plan with `write_todos` |
| 2 | **Delegate** | Parallelize investigation across subagents (up to 3 at once) |
| 3 | **Synthesize** | Combine the gathered information |
| 4 | **Verify** | Ask the fact-checker to verify the claims |
| 5 | **Report** | Produce the final report |


In [ ]:
from prompts import RESEARCH_AGENT_PROMPT

print(RESEARCH_AGENT_PROMPT)


## Step 4: Define Three Subagents

The deep research agent uses three specialized subagents:

| Subagent | Role | Tools |
|------------|------|------|
| `researcher-1` | Primary topic research | `web_search`, `think_tool` |
| `researcher-2` | Complementary or contrasting research | `web_search`, `think_tool` |
| `fact-checker` | Verify factual accuracy | `web_search` |


In [ ]:
researcher_1 = {
    "name": "researcher-1",
    "description": "Performs in-depth research on the topic",
    "system_prompt": "You are a research specialist. Investigate the topic deeply and summarize the key findings. Reflect with think_tool after searching.",
    "tools": [web_search, think_tool],
}


In [ ]:
researcher_2 = {
    "name": "researcher-2",
    "description": "Performs complementary research from another perspective",
    "system_prompt": "You are a complementary researcher. Collect additional information from a different angle. Reflect with think_tool after searching.",
    "tools": [web_search, think_tool],
}


In [ ]:
fact_checker = {
    "name": "fact-checker",
    "description": "Verifies the factual correctness of collected information",
    "system_prompt": "You are a fact-checker. Verify the accuracy of the provided information and point out any errors.",
    "tools": [web_search],
}


## Step 5: Create the Deep Research Agent (with v1 Middleware)

Combine the tools and subagents into the final agent. The v1 middleware improves stability and reliability:

| Middleware | Role |
|---------|------|
| `SummarizationMiddleware` | Automatically summarizes long research conversations to save context |
| `ModelCallLimitMiddleware` | Prevents research loops by limiting the run to 30 model calls |
| `ModelFallbackMiddleware` | Falls back to a backup model if the primary model fails |

Enable checkpointing with `InMemorySaver` so interrupted research runs can resume.


In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import (
    SummarizationMiddleware,
    ModelCallLimitMiddleware,
    ModelFallbackMiddleware,
)

research_agent = create_deep_agent(
    model=model,
    tools=[web_search, think_tool],
    subagents=[researcher_1, researcher_2, fact_checker],
    system_prompt=RESEARCH_AGENT_PROMPT,
    backend=FilesystemBackend(root_dir=".", virtual_mode=True),
    skills=["/skills/"],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(model=model, trigger=("messages", 15)),
        ModelCallLimitMiddleware(run_limit=30),
        ModelFallbackMiddleware("gpt-4.1-mini"),
    ],
)


## Step 6: Run the Research Workflow

Give the agent a research topic and let it execute the five-step workflow.


In [ ]:
thread = {"configurable": {"thread_id": "research-1"}}
response = research_agent.invoke(
    {"messages": [{"role": "user", "content": "Research the current trends in AI agent frameworks. Focus on LangGraph and Deep Agents, and provide a comparison."}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)


## Step 7: Streaming — Track Namespaces

With `stream(subgraphs=True)`, you can follow the execution of the main agent and the subagents by namespace. This makes it easy to see when each subagent is called.


In [ ]:
thread2 = {"configurable": {"thread_id": "research-2"}}
chunks = []
for ns, chunk in research_agent.stream(
    {"messages": [{"role": "user", "content": "Research the subagent architecture of Deep Agents."}]},
    config={**thread2, **lf_config},
    subgraphs=True,
):
    chunks.append((ns, chunk))

# Count chunks by namespace
from collections import Counter
ns_counts = Counter(ns for ns, _ in chunks)
print(f"Received {len(chunks)} chunks in total", flush=True)
for ns_name, cnt in ns_counts.items():
    label = ns_name if ns_name else "main"
    print(f"  [{label}] {cnt} chunks", flush=True)


## Subagent Design Best Practices

| Principle | Description |
|------|------|
| **Clear descriptions** | Write specific `description` values so the main agent knows when to delegate |
| **Specialized prompts** | Put output format, constraints, and workflow expectations into `system_prompt` |
| **Minimal tools** | Give each subagent only the tools it actually needs |
| **Concise outputs** | Tell subagents to return summaries rather than raw data |


## Summary

| Item | Key Point |
|------|------|
| **`think_tool`** | Strategic reflection — analyze search results and plan the next step |
| **Subagents** | Parallel execution with `researcher-1`, `researcher-2`, and `fact-checker` |
| **Workflow** | Plan → Delegate → Synthesize → Verify → Report |
| **Context management** | Only subagent results are passed back to the main agent; intermediate work stays isolated |

---

**References:**
- `docs/deepagents/examples/02-deep-research.md`
- `docs/deepagents/07-subagents.md`
- `docs/deepagents/06-backends.md`

**Previous Step:** ← [04_ml_agent.ipynb](./04_ml_agent.ipynb): Machine learning agent
